In [23]:
!pip install elasticsearch



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
#import necessary modules
import json
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

In [29]:
# check if the elasticsearch container is running
!curl http://localhost:9200/

{
  "name" : "STELLITSA",
  "cluster_name" : "elasticsearch",
  "cluster_uuid" : "PXP4MnYLQf6HHECQcMQSPg",
  "version" : {
    "number" : "9.2.0",
    "build_flavor" : "default",
    "build_type" : "zip",
    "build_hash" : "25d88452371273dd27356c98598287b669a03eae",
    "build_date" : "2025-10-21T10:06:21.288851013Z",
    "build_snapshot" : false,
    "lucene_version" : "10.3.1",
    "minimum_wire_compatibility_version" : "8.19.0",
    "minimum_index_compatibility_version" : "8.0.0"
  },
  "tagline" : "You Know, for Search"
}


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100   533  100   533    0     0   3504      0 --:--:-- --:--:-- --:--:--  3506


In [30]:
# instantiate Elasticsearch client localhost
client = Elasticsearch("http://localhost:9200")

In [32]:
# data are in a jsonl file
# each line is a different json record
# read jsonl file line by line

all_json_data = [] # a list of json records
with open("movies.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        json_data = json.loads(line)
        all_json_data.append(json_data)


In [33]:
# take a peek at the data
print(f" Datatype all_json_data: {type(all_json_data)}")
print(f" Datatype json_data: {type(json_data)}")
print(f" Length of the list: {len(all_json_data)}")
all_json_data[0]

#for jd in all_json_data:
#    print(f"Title: {jd['title']}, Runtime: {jd['runtime']}, Plot: {jd['plot']}, Keyscene: {jd['keyScene']} Genre: {jd['genre']}, Released: {jd['released']}")


 Datatype all_json_data: <class 'list'>
 Datatype json_data: <class 'dict'>
 Length of the list: 12


{'title': 'Pulp Fiction',
 'runtime': '154',
 'plot': 'The lives of two mob hitmen, a boxer, a gangster and his wife, and a pair of diner bandits intertwine in four tales of violence and redemption.',
 'keyScene': "John Travolta is forced to inject adrenaline directly into Uma Thurman's heart after she overdoses on heroin.",
 'genre': 'Crime, Drama',
 'released': '1994'}

In [51]:
# delete the index anytime
client.indices.delete(index="my_movies")

ObjectApiResponse({'acknowledged': True})

In [52]:
# configure index properties
# create fields

mapping = {
    "mappings": {
        "properties": {
            "title": {"type": "text"},
            "runtime": {"type": "integer"},
            "plot": {"type": "text"},
            "keyScene": {"type": "text"},
            "genre": {"type": "text"},
            "released": {"type": "date", "format": "yyyy"}
        }
    }
}

# create an index with the configuration above
client.indices.create(index='my_movies', body=mapping)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_movies'})

In [53]:
def insert_document(document):
    response = client.index(index="my_movies", body=document)
    return response

# populate index one document at a time
for document in all_json_data:
    response = insert_document(document)
    #print(document)

print("Done indexing documents into `my_movies` index!")

Done indexing documents into `my_movies` index!


In [54]:
# count documents in a specific index..<
client.cat.count(index=['my_movies'], params={"format": "json"})

C:\Users\moumt\AppData\Local\Temp\ipykernel_19876\2165790954.py:2: DeprecationWarning: The 'params' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  client.cat.count(index=['my_movies'], params={"format": "json"})


ListApiResponse([{'epoch': '1761504650', 'timestamp': '18:50:50', 'count': '12'}])

In [55]:
# index using bulk helpers
from elasticsearch.helpers import bulk

documents = []
for doc in all_json_data:
    documents.append({
        "_index": "my_movies",
        "_source": doc  # αν το doc είναι dict
    })

# Indexing με helpers.bulk
bulk(client, documents, refresh=True)

print("Done indexing documents into `my_movies` index!")

Done indexing documents into `my_movies` index!


In [56]:
def pretty_search_response(response):
    if len(response["hits"]["hits"]) == 0:
        print("Your search returned no results.")
    else:
        for hit in response["hits"]["hits"]:
            id = hit["_id"]
            score = hit["_score"]
            title = hit["_source"]["title"]
            runtime = hit["_source"]["runtime"]
            plot = hit["_source"]["plot"]
            keyScene = hit["_source"]["keyScene"]
            genre = hit["_source"]["genre"]
            released = hit["_source"]["released"]
            
            pretty_output = f"\nID: {id}\nScore: {score}\nTitle: {title}\nRuntime: {runtime}\nPlot: {plot}\nKeyScene: {keyScene}\nGenre: {genre}\nReleased: {released}"

            print(pretty_output)


In [57]:
search_results = client.search(index="my_movies", query={"match_all": {}})

# Print search results
pretty_search_response(search_results)



ID: gYPbIZoBq5SudV253P3C
Score: 1.0
Title: Pulp Fiction
Runtime: 154
Plot: The lives of two mob hitmen, a boxer, a gangster and his wife, and a pair of diner bandits intertwine in four tales of violence and redemption.
KeyScene: John Travolta is forced to inject adrenaline directly into Uma Thurman's heart after she overdoses on heroin.
Genre: Crime, Drama
Released: 1994

ID: goPbIZoBq5SudV253P3W
Score: 1.0
Title: The Dark Knight
Runtime: 152
Plot: When the menace known as the Joker wreaks havoc and chaos on the people of Gotham, Batman must accept one of the greatest psychological and physical tests of his ability to fight injustice.
KeyScene: Batman angrily responds 'I’m Batman' when asked who he is by Falcone.
Genre: Action, Crime, Drama, Thriller
Released: 2008

ID: g4PbIZoBq5SudV253P3j
Score: 1.0
Title: Fight Club
Runtime: 139
Plot: An insomniac office worker and a devil-may-care soapmaker form an underground fight club that evolves into something much, much more.
KeyScene: Brad 

In [58]:
search_results = client.search(
    index="my_movies",
    size=5,
    query = {"match": {"plot": "cannibal killer"}},
)

# Print search results
pretty_search_response(search_results)

#experiment with 'killer', 'serial killer', 'cannibal killer'


ID: iYPbIZoBq5SudV253f0i
Score: 4.5355797
Title: The Silence of the Lambs
Runtime: 118
Plot: A young F.B.I. cadet must receive the help of an incarcerated and manipulative cannibal killer to help catch another serial killer, a madman who skins his victims.
KeyScene: Hannibal Lecter explains to Clarice Starling that he ate a census taker's liver with some fava beans and a nice Chianti.
Genre: Crime, Drama, Thriller
Released: 1991

ID: lYPbIZoBq5SudV258P2V
Score: 4.5355797
Title: The Silence of the Lambs
Runtime: 118
Plot: A young F.B.I. cadet must receive the help of an incarcerated and manipulative cannibal killer to help catch another serial killer, a madman who skins his victims.
KeyScene: Hannibal Lecter explains to Clarice Starling that he ate a census taker's liver with some fava beans and a nice Chianti.
Genre: Crime, Drama, Thriller
Released: 1991

ID: iIPbIZoBq5SudV253f0X
Score: 1.8676022
Title: Se7en
Runtime: 127
Plot: Two detectives, a rookie and a veteran, hunt a serial kil

In [59]:
print("English analyzer:")
response = client.indices.analyze(
    body={
        "analyzer": "english",
        "text": "The actors were running quickly across the stage!"
    }
)

for token in response["tokens"]:
    print(token["token"])

print("\nStandard analyzer:")
response = client.indices.analyze(
    body={
        "analyzer": "standard",
        "text": "The actors were running quickly across the stage!"
    }
)

for token in response["tokens"]:
    print(token["token"])    

English analyzer:
actor
were
run
quickli
across
stage

Standard analyzer:
the
actors
were
running
quickly
across
the
stage


In [45]:
# delete the index anytime
client.indices.delete(index="my_movies")

ObjectApiResponse({'acknowledged': True})

In [62]:
# set similarity function to vector space model (tfidf)
# set analyzer to english analyzer (stopword removal, stemming, puncuation removal, etc.) for indexing and querying

vsm_mapping = {
    "settings": {
        "number_of_shards": 1,
        "similarity": {
            "scripted_tfidf": {
                "type": "scripted",
                "script": {
                    "source": "double tf = Math.sqrt(doc.freq); double idf = Math.log((field.docCount+1.0)/(term.docFreq+1.0)) + 1.0; double norm = 1/Math.sqrt(doc.length); return query.boost * tf * idf * norm;"
                }
            }
        },        
        "analysis": {
            "analyzer": {
                "default": {
                    "type": "english"
                },
                "default_search": {
                    "type": "english"
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "title": {
                "type": "text",
                "similarity": "scripted_tfidf"
            },
            "plot": {
                "type": "text",
                "similarity": "scripted_tfidf"
            },
            "keyScene": {
                "type": "text",
                "similarity": "scripted_tfidf"
            },
            "runtime": {
                "type": "integer"
            },
            "genre": {
                "type": "text",
                "similarity": "scripted_tfidf"
            },
            "released": {
                "type": "date",
                "format": "yyyy"
            }
        }
    }
}

# create an index with the configuration above
client.indices.create(index='my_movies', body=vsm_mapping)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_movies'})

In [64]:
# delete the index anytime
client.indices.delete(index="my_movies")

ObjectApiResponse({'acknowledged': True})

In [65]:
# set similarity function to vector space model (tfidf)
# set analyzer to english analyzer (stopword removal, stemming, puncuation removal, etc.) for indexing and querying

vsm_optimized_mapping = {
    "settings": {
        "similarity": {
            "scripted_tfidf": {
                "type": "scripted",
                "script": {
                    "source": "double tf = Math.sqrt(doc.freq); double idf = Math.log((field.docCount+1.0)/(term.docFreq+1.0)) + 1.0; double norm = 1/Math.sqrt(doc.length); return query.boost * tf * idf * norm;"
                }
            }
        },        
        "analysis": {
            "analyzer": {
                "default": {
                    "type": "english"
                },
                "default_search": {
                    "type": "english"
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "title": {
                "type": "text",
                "copy_to": "allContent",
                "similarity": "scripted_tfidf"
            },
            "plot": {
                "type": "text",
                "copy_to": "allContent",
                "similarity": "scripted_tfidf"
            },
            "keyScene": {
                "type": "text",
                "copy_to": "allContent",                
                "similarity": "scripted_tfidf"
            },
            "runtime": {
                "type": "integer"
            },
            "genre": {
                "type": "text",
                "copy_to": "allContent",                
                "similarity": "scripted_tfidf"
            },
            "released": {
                "type": "date",
                "format": "yyyy"
            },
            "allContent": {
                "type": "text",
                "similarity": "scripted_tfidf"                
            }            
        }
    }
}

# create an index with the configuration above
client.indices.create(index='my_movies', body=vsm_optimized_mapping)


ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'my_movies'})

In [66]:
search_results = client.search(
    index="my_movies",
    size=5,
    query = {"match": {"allContent": "killer"}},
)

# Print search results
pretty_search_response(search_results)

Your search returned no results.


In [67]:
from pprint import pprint
# The search results are in the field 'hits':
print("Total articles found", search_results['hits']['total']) #total results count returned
# Another way to check how many results have been actually returned:
print("Results returned", len(search_results['hits']['hits'])) # the response was limited to top 10 results
# Let's access the first result:
pprint(search_results['hits']['hits'][0]) #use pprint for a more convenient display of hierarchical structure

Total articles found {'value': 0, 'relation': 'eq'}
Results returned 0


IndexError: list index out of range

In [ ]:
search_results = client.search(
    index="my_movies",
    size=5,
    query = {"multi_match": {"query": "police", "fields": ["plot", "keyScene"]}},
)

# Print search results
pretty_search_response(search_results)

In [ ]:
search_results = client.search(
    index="my_movies",
    size=5,
    query = {"multi_match": {"query": "police", "fields": ["plot", "keyScene^3"]}},
)

# Print search results
pretty_search_response(search_results)

In [ ]:
search_results = client.search(
    index="my_movies", query={"range": {"runtime": {"gte": 150}}}
)

# Print search results
pretty_search_response(search_results)



In [ ]:
search_results = client.search(
    index="my_movies", query={"prefix": {"title": {"value": "go"}}}
)

# Print search results
pretty_search_response(search_results)

In [ ]:
search_results = client.search(
    index="my_movies",
    query={
        "bool": {
            "must": [
                {"match": {"plot": "crime"}},
                {"match": {"keyScene": "character"}},
            ]
        }
    },
)

# Print search results
pretty_search_response(search_results)



In [ ]:
search_results = client.search(
    index="my_movies",
    query={
        "bool": {
            "should": [
                {"match": {"plot": "crime"}},
                {"match": {"keyScene": "character"}},
            ]
        }
    },
)

# Print search results
pretty_search_response(search_results)
